In [2]:
import pandas as pd
from pathlib import Path
from _collections import defaultdict

def read_run(path):
    return pd.read_csv(
        path,
        sep=r"\s+",
        names=["qid", "Q0", "doc_id", "rank", "score", "system"]
    )

bm25 = read_run("runs/bm25.trec")
knn = read_run("runs/knn.trec")

k_rrf = 60
scores = defaultdict(float)

for df in [bm25, knn]:
    for _, row in df.iterrows():
        key = (row["qid"], row["doc_id"])
        scores[key] += 1 / (k_rrf + row["rank"])

rows = []

for (qid, doc_id), score in scores.items():
    rows.append({
        "qid": qid,
        "doc_id": doc_id,
        "score": score
    })

hybrid = pd.DataFrame(rows)

Path("runs").mkdir(exist_ok=True)

with open("runs/hybrid_rrf.trec", "w", encoding="utf-8") as f:
    for qid, group in hybrid.groupby("qid"):
        group = group.sort_values("score", ascending=False).head(100)

        for rank, (_, row) in enumerate(group.iterrows(), 1):
            f.write(
                f"{qid} Q0 {row["doc_id"]} {rank} {row['score']:.6f} hybrid_rrf\n"
            )

print("Run híbrido salvo em runs/hybrid_rrf.trec")

Run híbrido salvo em runs/hybrid_rrf.trec
